# N20 · CUDA Graph capture/replay 直觉

> 配套 `labs/l30.5_cuda_graph_savor/`。CPU 模拟，但概念与真实 CUDA Graph 一致。
>
> 跑完后你应能解释：
> 1. 为什么 decode loop 是 CUDA Graph 的 sweet spot，prefill 不是
> 2. capture 时输入张量地址固定为什么很重要
> 3. 不同 batch size 为什么需要分别 capture


## 1. 朴素直觉：launch overhead 不是免费的

GPU kernel launch 本身有 5–20μs 的 CPU→GPU 提交开销。如果一个 decode step 总共调用 50 个 kernel，纯 launch overhead 就是 250μs–1ms——而 decode 本身可能也就这么久。

CUDA Graph 把 N 个 kernel 的依赖图一次性提交，replay 时 GPU 端按图自跑，CPU 几乎 idle。


In [ ]:
import time
import torch

def simulate_decode_step(x):
    # 50 个细碎 op，模拟 attention + MLP + RMSNorm 等
    for _ in range(50):
        x = x * 1.0001 + 0.0001
    return x

x = torch.randn(64, 1024)

N_STEPS = 200
start = time.perf_counter()
for _ in range(N_STEPS):
    x = simulate_decode_step(x)
eager_ms = (time.perf_counter() - start) * 1000
print(f"eager: {eager_ms:.2f} ms for {N_STEPS} steps  ({eager_ms/N_STEPS:.3f} ms/step)")

In [ ]:
# 概念上等价于 graph.replay：把整个序列封装成一个函数指针，调用时 CPU 几乎只做 dispatch
import time, torch

x = torch.randn(64, 1024)

# 'capture'：把 50 op 折叠成一个可重用对象（这里就是函数本身）
captured = simulate_decode_step  # 真实 CUDA：torch.cuda.graph(...)

start = time.perf_counter()
for _ in range(N_STEPS):
    x = captured(x)
replay_ms = (time.perf_counter() - start) * 1000
print(f"'replay' (single dispatch): {replay_ms:.2f} ms")
print("实际 CUDA Graph 上的 speedup 通常是 1.3x-3x，取决于 op 数与 batch 大小")

## 2. 为什么 prefill 不适合

每个 prefill 请求长度都不同：
- bs=1 prompt 长度 128 → 一种 shape
- bs=1 prompt 长度 256 → 另一种 shape
- bs=4 各自不同 → 又一种

`GraphCache` 必须按 shape 分别 capture。Prompt 长度分布越广，cache 命中率越低。生产里 SGLang/vLLM 给 prefill 用 chunked prefill 把多种长度归到固定 chunk size 上，再考虑 graph。

而 decode 永远是 `(bs, 1)`，shape 集中在 `bs ∈ {1, 2, 4, 8, ...}` 几个值，graph cache 命中率接近 100%。


In [ ]:
from collections import defaultdict
import torch

class GraphCache:
    def __init__(self):
        self._graphs = {}
        self.capture = 0
        self.replay = 0
    def __call__(self, fn, x):
        key = tuple(x.shape)
        if key not in self._graphs:
            self._graphs[key] = fn
            self.capture += 1
            return fn(x)
        self.replay += 1
        return self._graphs[key](x)

gc = GraphCache()
decode_op = lambda t: t * 2

# decode：bs ∈ {1, 2, 4}，反复跑
for _ in range(100):
    for bs in [1, 2, 4]:
        gc(decode_op, torch.randn(bs, 1, 1024))
print(f"decode (bs=1,2,4 each 100x): captures={gc.capture}, replays={gc.replay}")

gc2 = GraphCache()
prefill_op = lambda t: t * 2
# prefill：长度从 128 到 1024 各种
import random
random.seed(0)
for _ in range(100):
    seq_len = random.randint(64, 1024)
    gc2(prefill_op, torch.randn(1, seq_len, 1024))
print(f"prefill (random seq_len): captures={gc2.capture}, replays={gc2.replay}")

## 3. 自检 / 面试题

1. CUDA Graph 在 prefill 上为什么收益小？（shape 离散度高，cache 命中低）
2. SGLang `torch_memory_saver.pause` 释放显存后，已经 capture 的 graph 还能 replay 吗？（不能直接 replay；需要先 resume 把虚拟地址背后的物理 page 接回，否则 graph 里指向的指针失效）
3. 真实 CUDA Graph 的 capture 阶段为什么要 warmup？（首次跑可能触发 cudnn benchmark / 算子选型，必须 warmup 之后再 capture，否则 capture 进去的是次优 kernel）
4. 如何处理 batch size 列表如 `[1, 2, 4, 8, 16, 32]`？（按 bs 分别 capture，padding 到最近的 bucket）
